# SNUC GLOFeagles 2026 Challenge
## Glacial Lake Detection – Reproducibility Pipeline Notebook

This notebook demonstrates how to load the trained custom U-Net model and run inference on sample satellite imagery to segment glacial lakes.

In [ ]:
import os
import torch
import cv2
import matplotlib.pyplot as plt
import numpy as np
from model import build_model
from utils import load_image, overlay_mask
import config

print("PyTorch version:", torch.__version__)
print("Configured Device:", config.DEVICE)

### 1. Load the Model from Checkpoint

In [ ]:
# Define paths
checkpoint_path = os.path.join("outputs", "checkpoints", "best_model.pth")

# Build model and load weights
model = build_model(pretrained=False)
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=config.DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    print(f"Successfully loaded best model from epoch {checkpoint.get('epoch', 'N/A')}")
else:
    print(f"[WARNING] Checkpoint not found at: {checkpoint_path}. Please make sure you have run the training pipeline first.")

### 2. Define Inference Function

In [ ]:
def run_inference(image_path, model, device, threshold=0.45):
    # Load and preprocess image
    orig_img = load_image(image_path)
    h, w, _ = orig_img.shape
    
    # Resize and normalize
    img_resized = cv2.resize(orig_img, (512, 512))
    x = img_resized.astype(np.float32) / 255.0
    # ImageNet mean & std normalization
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    x = (x - mean) / std
    x = np.transpose(x, (2, 0, 1)) # HWC to CHW
    x = torch.tensor(x).unsqueeze(0).to(device) # Add batch dimension
    
    # Forward pass
    with torch.no_grad():
        logits = model(x)
        probs = torch.sigmoid(logits).squeeze().cpu().numpy()
    
    # Resize prediction back to original size
    probs_orig = cv2.resize(probs, (w, h), interpolation=cv2.INTER_LINEAR)
    
    # Threshold and morphological post-processing
    binary_mask = (probs_orig > threshold).astype(np.uint8) * 255
    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_OPEN, kernel_open)
    binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_CLOSE, kernel_close)
    
    return orig_img, binary_mask

### 3. Run on a Sample Image and Visualize

In [ ]:
# Find a sample image in the dataset directory
dataset_dir = os.path.join("..", "SNUC GLOFeagles 2026 challenge datasets")
if os.path.exists(dataset_dir):
    images = [f for f in os.listdir(dataset_dir) if f.endswith(".png")]
    if images:
        sample_img_path = os.path.join(dataset_dir, images[0])
        print(f"Using sample image: {sample_img_path}")
        
        # Run inference
        orig, mask = run_inference(sample_img_path, model, config.DEVICE)
        overlay = overlay_mask(orig, mask, color=(0, 100, 255), alpha=0.4)
        
        # Plotting
        plt.figure(figsize=(15, 5))
        
        plt.subplot(1, 3, 1)
        plt.imshow(orig)
        plt.title("Original Image")
        plt.axis("off")
        
        plt.subplot(1, 3, 2)
        plt.imshow(mask, cmap="gray")
        plt.title("Predicted Glacial Lake Mask")
        plt.axis("off")
        
        plt.subplot(1, 3, 3)
        plt.imshow(overlay)
        plt.title("Overlay Result")
        plt.axis("off")
        
        plt.tight_layout()
        plt.show()
    else:
        print("No images found in dataset directory!")
else:
    print(f"Dataset directory not found at: {dataset_dir}")